<h1><b><center>Jackpot Project</center></b></h1>

<h3><b><center>Project Brief</center></b></h3>

Identify and retrieve the Player IDs of players (excluding fraudulent accounts) who participated consistently in jackpot events 9582, 9583, 9584, and 9585. Only players who did not miss any week are required.

**Cases**
1. Identify and retrieve the Player IDs of players (excluding fraudulent accounts) who took part in all jackpots. 

2. Identify and retrieve the Player IDs of players (excluding fraudulent accounts) who started playing in the second week (Jackpot event ID 9583) and played consistently through the fourth week, i.e., Jackpots 9583, 9584, and 9585.

3. Identify and retrieve the Player IDs of players (excluding fraudulent accounts) who started playing in the third week (Jackpot event ID 9584) and played consistently through the fourth week, i.e., Jackpots 9584 and 9585.

4. Identify and retrieve the Player IDs of players (excluding fraudulent accounts) who took part in jackpot 9584 but did not participate in jackpot 9585. 

<h3><b><center>Code & Logic</center></b></h3>

**1. Import Packages**

In [ ]:
# Package used to connect to MySQL Databases
import mysql.connector
import pymysql
import paramiko
from sqlalchemy import create_engine


# Folder Creation
import os
from pathlib import Path
from dotenv import load_dotenv

# Data Manipulation Packages
import pandas as pd
import numpy as np
from google.cloud import bigquery
import pandas as pd

# Package To Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

**2. Get Credentials From .env File**

In [ ]:
ROOT_DIR: Path = Path().resolve()
load_dotenv(os.path.join(ROOT_DIR, ".env"))
Root = os.path.normpath(os.getcwd() + os.sep + os.pardir)

In [ ]:
host_ = os.getenv('HOST')
database_ = os.getenv('DATABASE')
user_ = os.getenv('NAME')
password_ = os.getenv('PASSWORD')
port_ = os.getenv('PORT')

In [ ]:
destination_connection = "BI"
destination_table_name = "jackpot_bets"
destination_schema     = "bi_schema"
engine_string          = 'mysql+pymysql://'+user_+':'+password_+'@'+host_+':'+port_+'/'+destination_schema

**3. Create SQL Query For Big Query To Get player_ids**

In [ ]:
BQ = """
      WITH jackpots AS
            (SELECT a.bet_id 
                  ,b.player_id
                  ,a.jackpot_event_id
            FROM `project-bi.bi_schema.jackpot_ids` AS a
            LEFT JOIN `project-bi.bi_schema.km_jackpot_transactions` AS b
            ON CAST(a.bet_id AS INT) = CAST(b.bet_id AS INT)
            ),

      counts AS 
            (SELECT player_id
            ,SUM(CASE WHEN jackpot_event_id IN (9582) THEN 1 ELSE 0 END) AS _9582
            ,SUM(CASE WHEN jackpot_event_id IN (9583) THEN 1 ELSE 0 END) AS _9583
            ,SUM(CASE WHEN jackpot_event_id IN (9584) THEN 1 ELSE 0 END) AS _9584
            ,SUM(CASE WHEN jackpot_event_id IN (9585) THEN 1 ELSE 0 END) AS _9585
            FROM jackpots
            GROUP BY 1)

      SELECT * FROM counts;
      """

**4. Import Data as Dataframe From Big Query**

In [ ]:
df = pd.read_gbq(BQ, project_id='project-bi')

**5. Count player_ids Before Removal Of Fradualant Accounts**

In [ ]:
df['player_id'].count()

**6. Import Fradualant Accounts From MySQL**

In [ ]:
MS = 'SELECT DISTINCT player_id FROM bi_schema_local.fradualant_accounts' 

In [ ]:
# Code To Connect MySQL
connection = mysql.connector.connect(host=host_
                                      ,database=database_
                                      ,user=user_
                                      ,password=password_
                                      ,port=port_)

# Connect to MySQL database
try:
    with connection.cursor() as cursor:
        ex = pd.read_sql(MS,connection)
finally:
    connection.close()

In [ ]:
df.head()

**6. Remove Fradualant Accounts**

In [ ]:
ids_to_remove = set(ex['player_id'])
df = df[~df['player_id'].isin(ids_to_remove)]

**7. Count player_ids After Removal Of Fradualant Accounts**

In [ ]:
df['player_id'].count()

**8. Create Logic To Find Players For The Different Cases Stated In The Breif**

In [ ]:
df['case'] = df.apply(
    lambda row: 'case 1' if row['_9582'] >= 1 and row['_9583'] >= 1 and row['_9584'] >= 1 and row['_9585'] >= 1
    else 'case 2' if row['_9582'] == 0 and row['_9583'] >= 1 and row['_9584'] >= 1 and row['_9585'] >= 1
    else 'case 3' if row['_9582'] == 0 and row['_9583'] == 0 and row['_9584'] >= 1 and row['_9585'] >= 1
    else 'case 4' if row['_9582'] == 0 and row['_9583'] == 0 and row['_9584'] >= 1 and row['_9585'] >= 0
    else 'none',
    axis=1
)

**9. Export To Excel**

In [ ]:
with pd.ExcelWriter('MBW Jackpot Players V3.xlsx') as writer:  
    
    df[df['case']=='case 1'].to_excel(writer, sheet_name='Case 1', index=False)
    df[df['case']=='case 2'].to_excel(writer, sheet_name='Case 2', index=False)
    df[df['case']=='case 3'].to_excel(writer, sheet_name='Case 3', index=False)
    df[df['case']=='case 4'].to_excel(writer, sheet_name='Case 4', index=False)

<h3><b><center>Outcome</center></b></h3>

1. Pulled Data From Big Query
2. Pulled Data From MySQL Database
3. Compared Data between both data sources, and exluded specified data.
4. Used if statement to partition data
5. Export Data into excel file with multiple tabs